# 🚀 Notebook 04 — Sluttanbefaling

**Det er møtetid. Marte venter på svaret ditt.**

Popularitet ga oss en baseline, metadata ga oss cold-start-støtte, ALS ga oss styrke,
og hybridtenkningen ga oss produktrealismen. Nå samler vi trådene og svarer:
**hva bør StreamNord faktisk shippe?**

## Oppsett

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.sparse import lil_matrix
from src.data import load_interactions, load_item_metadata, get_genre_matrix
from src.split import leave_one_out_split, build_sparse_matrix
from src.metrics import recall_at_k, ndcg_at_k, map_at_k, coverage, novelty
from src.rerank import mmr_rerank
from src.recommenders.popularity import PopularityRecommender
from src.recommenders.item_item import ItemItemRecommender
from src.recommenders.als import ALSRecommender
from src.recommenders.content_based import ContentBasedRecommender

interactions = load_interactions()
items = load_item_metadata()
train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
genre_matrix = get_genre_matrix(items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10
item_pop = np.asarray(train_matrix.sum(axis=0)).flatten() / n_users
LEA_ID = 451

print('Oppsett ferdig')

## 🏋️ Oppgave 6 — Leaderboard for modellfamiliene

Alle modellene side om side. Se ikke bare på Recall —
sjekk coverage og novelty. Hva forteller de?

In [ ]:
pop = PopularityRecommender().fit(train_matrix)
item_item = ItemItemRecommender().fit(train_matrix)
als = ALSRecommender(factors=64).fit(train_matrix)
content = ContentBasedRecommender().fit(genre_matrix)

models = {
    'Popularitet': pop.recommend(user_ids, train_matrix, K),
    'Innholdsbasert': content.recommend(user_ids, train_matrix, K),
    'Item-item': item_item.recommend(user_ids, train_matrix, K),
    'ALS': als.recommend(user_ids, train_matrix, K),
}

cand_ids, cand_scores = als.model.recommend(user_ids, train_matrix[user_ids], N=50, filter_already_liked_items=True)
recs_mmr = np.zeros((len(user_ids), K), dtype=np.int32)
for row_index in range(len(user_ids)):
    recs_mmr[row_index] = mmr_rerank(cand_ids[row_index], cand_scores[row_index], genre_matrix, k=K, lambda_=0.6)
models['ALS+MMR'] = recs_mmr

results = {}
for name, recs in models.items():
    results[name] = {
        f'recall@{K}': recall_at_k(recs, test_items, K),
        f'ndcg@{K}': ndcg_at_k(recs, test_items, K),
        f'map@{K}': map_at_k(recs, test_items, K),
        f'coverage@{K}': coverage(recs, n_items, K),
        f'novelty@{K}': novelty(recs, item_pop, K),
    }

leaderboard = pd.DataFrame(results).T.sort_values(f'ndcg@{K}', ascending=False)
print(leaderboard.to_string(float_format=lambda value: f'{value:.4f}'))

## 📋 Beslutningsmal

Fyll inn malen nedenfor før du skriver anbefalingen til Marte.
Ta den med deg etter workshopen — den er gjenbrukbar.

In [ ]:
# BESLUTNINGSMAL — Anbefalingssystem
#
# 1. SIGNALTYPE
#    Hovedsakelig eksplisitt / implisitt / begge: 
#
# 2. ANBEFALT MODELLFAMILIE
#    Kjernemodell: 
#    Hvorfor denne: 
#
# 3. COLD-START-STRATEGI
#    For nye brukere: 
#    For nye items: 
#
# 4. FAIRNESS OG MANGFOLD
#    Hovedrisiko: 
#    Tiltak: 
#
# 5. PRODUKSJONSARKITEKTUR
#    Kandidatgenerering: 
#    Rangering: 
#    Re-rangering: 
#
# 6. KJENTE RISIKOER
#    Hva kan gå galt: 
#    Hva mangler vi data for: 


## 🏋️ Oppgave 7 — Hva anbefaler du å shippe?

Bruk malen du nettopp fylte ut. Skriv en kort anbefaling til Marte — ikke tenk bare
på hva som vinner én metrikk, men på hvilket system som faktisk er mest realistisk.

> **Skriv din anbefaling her**
>
> *«Marte, basert på analysen anbefaler jeg ... fordi ...»*

## 🏋️ Oppgave 8 — Cold start

Hva skjer når en ny bruker dukker opp — og ALS har *ingenting* å jobbe med?

In [ ]:
def evaluate_cold_start(als_rec, train_matrix, test_df, history_sizes=(1, 3, 5, 10, 20, 50), k=10):
    rng = np.random.default_rng(42)
    user_ids = test_df['user_id'].values
    test_items = test_df['item_id'].values
    rows = []
    for history_size in history_sizes:
        masked = lil_matrix(train_matrix.shape, dtype=np.float32)
        for user_id in user_ids:
            seen = train_matrix[user_id].indices
            keep = seen if len(seen) <= history_size else rng.choice(seen, size=history_size, replace=False)
            masked[user_id, keep] = 1.0
        masked_csr = masked.tocsr()
        recs = als_rec.model.recommend(user_ids, masked_csr[user_ids], N=k, filter_already_liked_items=True, recalculate_user=True)[0]
        rows.append({'history_size': history_size, f'recall@{k}': recall_at_k(recs, test_items, k)})
    return rows

cold_start_rows = evaluate_cold_start(als, train_matrix, test_df)
cold_start_df = pd.DataFrame(cold_start_rows)
print(cold_start_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cold_start_df['history_size'], cold_start_df[f'recall@{K}'], 'o-', linewidth=2)
ax.set_xlabel('Antall interaksjoner i historikk')
ax.set_ylabel(f'Recall@{K}')
ax.set_title('Cold-start: ALS-ytelse vs historikkstørrelse')
plt.tight_layout()
plt.show()

## Produksjonsarkitektur

```
Kandidatgenerering  ->  Rangering  ->  Re-rangering
  (enkle signaler)      (sterk modell)     (fairness/regler)
```

### En realistisk anbefaling

- bruk **ALS** eller en tilsvarende sterk collaborative modell som hovedmotor
- bruk **innholdsbaserte signaler** for cold start og forklarbarhet
- bruk **reranking** for mangfold, fairness og produktkrav
- legg til **kontekst** når det gir tydelig verdi

## 🎬 Leas reise gjennom workshopen

In [ ]:
lea_idx = np.where(user_ids == LEA_ID)[0]
if len(lea_idx) > 0:
    print('Leas anbefalinger gjennom workshopen:')
    print('=' * 50)
    for name in ['Popularitet', 'Innholdsbasert', 'ALS', 'ALS+MMR']:
        lea_recs = models[name][lea_idx[0]]
        titles = items.set_index('item_id').loc[lea_recs[:5], 'title'].values
        print(f'\n{name}:')
        for rank, title in enumerate(titles, 1):
            print(f'  {rank}. {title}')
    print('\n-> Fra mainstream-spam til en mer personlig og balansert liste.')

### 🔄 Refleksjon — sjekk gjetningene fra starten

Gå tilbake til notebook 00 og les gjetningene du skrev ned.

> 1. Hva hadde du rett i?
> 2. Hva tok du feil om?
> 3. Hva overrasket deg mest gjennom hele workshopen?

## 🔑 Oppsummering

| # | Lærdom |
|---|--------|
| 1 | **Signalet betyr noe** — implisitt data må tolkes annerledes enn eksplisitt feedback |
| 2 | **Baselines er viktige** — popularitet er enkel, sterk og utilstrekkelig |
| 3 | **Metadata hjelper tidlig** — content-based filtering er nyttig, men begrenset |
| 4 | **Collaborative filtering skaper et hopp** — ALS lærer struktur metadata ikke ser |
| 5 | **Hybrider vinner i praksis** — produktkrav tvinger frem flere signaler |
| 6 | **Fairness må måles** — høy gjennomsnittlig relevans er ikke nok |

### Ressurser

- Hu, Koren & Volinsky (2008): *Collaborative Filtering for Implicit Feedback*
- Koren, Bell & Volinsky (2009): *Matrix Factorization Techniques*
- Sarwar et al. (2001): *Item-based Collaborative Filtering Recommendation Algorithms*
- He et al. (2017): *Neural Collaborative Filtering*
- Steck (2018): *Calibrated Recommendations*
- Abdollahpouri et al. (2020): *Popularity Bias in Recommendation*